In [148]:
import pandas as pd
import numpy as np
import sys
import warnings

warnings.filterwarnings("ignore")

# Reload validation_helpers if it has been edited
if "validation_helpers" in sys.modules:
    del sys.modules["validation_helpers"]
if "config" in sys.modules:
    del sys.modules["config"]


from validation_helpers import (
    build_predictor_features,
    clean_measure_values,
    fill_dummy_measure_values,
)

from config import (
    main_cols,
    WP4_DATASET_PATH,
    OUTPUT_DIR
)

print(WP4_DATASET_PATH)

import validation_helpers

print(validation_helpers.__file__)

/Users/zachbrubert/reducehf/output/dataset_wp4.csv.gz
/Users/zachbrubert/reducehf/analysis/validation/./validation_helpers.py


## 0. Read WP4 Data

In [132]:
df = pd.read_csv(WP4_DATASET_PATH, compression="gzip")

## 0. Reduce the data

In [133]:
df = df[main_cols]

print ((df.columns))

Index(['patient_id', 'sex', 'ethnicity_cat', 'index_date', 'birth_date',
       'sysbp_value', 'bmi_value', 'last_cholesterol_value',
       'last_hdl_cholesterol_value', 'last_hba1c_value', 'smoking',
       'last_hypertension_date_med', 'last_diabetes_medication_date',
       'hf_diagnosis_date', 'hf_diagnosis_primary_date',
       'hf_diagnosis_apc_date', 'hf_diagnosis_ec_date',
       'fasting_blood_glucose_value', 'imd_quintile'],
      dtype='object')


## 1. Derive Features

In [134]:
df_features = build_predictor_features(df)



## 2. Fix Dummy Data

In [135]:
IS_DUMMY_DATA = os.getenv("OPENSAFELY_BACKEND") is None
if IS_DUMMY_DATA:

    df_features = fill_dummy_measure_values(
        df_features,
        missing_prop=0,
        seed=42
    )

## 3. Data Cleaning: Check Measures
Based on the link below and Joe's suggestions.  
https://github.com/Exeter-Diabetes/EHRBiomarkr/blob/main/data-raw/biomarker_acceptable_limits.yaml

In [136]:
df_features = clean_measure_values(df_features)

## 4. Check Missing Data

In [137]:
#Exclude features with >80% missing values 
print ("Before excluding features with >80% missing values: ", len(df_features.columns))

missing_vals = ["", " ", "NA", "N/A", "na", "n/a", "None", "none", "NULL", "null", "Unknown", "unknown"]

df_features = df_features.replace(missing_vals, np.nan)

missing_prop = df_features.isnull().mean()

dropped_cols = missing_prop[missing_prop > 0.8].index.tolist()

## df_features = df_features.loc[:, missing_prop <= 0.8]
##print("Dropped columns:", dropped_cols)
##print ("After excluding features with >80% missing values: ", len(df_features.columns))



Before excluding features with >80% missing values:  21


## 5. Transform to One-hot-encoding

In [138]:
# -------------------------
# Recode variables for PCP-HF validation
# -------------------------

df_features["sex"] = df_features["sex"].map({
    "male": 1,
    "female": 0
})

# Ethnicity
ethnicity = df_features["ethnicity_cat"].astype("string").str.lower()

for group in ["asian", "black", "white", "mixed", "other"]:
    df_features[f"ethnicity_{group}"] = (
        ethnicity.eq(group).fillna(False).astype(int)
    )

df_features["ethnicity_NA"] = ethnicity.isna().astype(int)

df_features = df_features.drop(columns="ethnicity_cat")


# Smoking: N = 0, Y = 1
df_features["smoking"] = (
    df_features["smoking"]
    .astype(str)
    .str.upper()
    .map({
        "N": 0,
        "Y": 1
    })
)

# HF diagnosis: date present = 1, missing = 0
hf_cols = [
    "hf_diagnosis_primary_date",
    "hf_diagnosis_ec_date",
    "hf_diagnosis_apc_date",
    "hf_diagnosis_date",
]

df_features[hf_cols] = df_features[hf_cols].apply(
    pd.to_datetime,
    errors="coerce"
)

df_features["hf_diagnosis"] = (
    df_features[hf_cols].notna().any(axis=1).astype(int)
)

df_features["hf_community"] = (
    df_features["hf_diagnosis_primary_date"].eq(
        df_features["hf_diagnosis_date"]
    )
    & df_features["hf_diagnosis_primary_date"].notna()
).astype(int)

# Check remaining missing values
missing_after_recode = df_features.isnull().mean()

missing_after_recode = (
    missing_after_recode[missing_after_recode > 0]
    .sort_values(ascending=False)
)

print("Columns still containing missing values:")
print(missing_after_recode)

Columns still containing missing values:
hf_diagnosis_primary_date    0.9884
imd_quintile                 0.8022
hf_diagnosis_ec_date         0.6572
hf_diagnosis_apc_date        0.5378
hf_diagnosis_date            0.1794
diasbp_value                 0.0030
smoking                      0.0004
dtype: float64


## 6. Save datasets

In [147]:

os.makedirs(OUTPUT_DIR, exist_ok=True)

path = os.path.join(
    OUTPUT_DIR,
    "DATA_VALIDATION_FULL.csv.gz"
)

df_features.to_csv(
    path,
    index=False,
    compression="gzip"
)

print(f"Saved: {path}")


Saved: /Users/zachbrubert/reducehf/output/validation/DATA_VALIDATION_FULL.csv.gz
